# Early AstroMiNN acquisition → later FLARE energy — November revision

**Question:** do objects with high early classification uncertainty also receive high anomaly energy once their light curves develop?

Default input: the four existing November 2025 BOOM/ZTF pools. We keep the **earliest saved snapshot per object** across these bounded queries, exclude known SSO associations using only the early packet, then freeze an entropy acquisition on that early universe. This is a new pooled archival experiment; it does not reproduce the four original per-pool top-100 selections. "Earliest saved" does not mean the first-ever alert, and individual objects have different snapshot times.

1. Query full currently available BOOM history **without a date cutoff**; cache it for every early object.
2. Audit whether any genuinely later measurements were retrieved. Diagnose coverage at 30, 60 and 100 days, and in the full retrieved archive.
3. Use **snapshot + 60 days** for the primary comparison. FLARE keeps its native first-100-day window from first usable detection; an old source's clock is never reset.
4. Compare early AstroMiNN entropy with FLARE `bts6.anomaly_energy` on the same eligible objects, including nonselections. Report the original acquisition's follow-up coverage separately from conditional ranking overlap.

Start with **`RUN_MODE = "coverage"`**. Then inspect coverage before changing it to `"compare"`. November's earlier applicability audit found some delayed eligibility; this is a reason to test it, not a guarantee of enough independent objects.

An optional **`INPUT_MODE = "fresh_boom"`** creates a new pool in fixed daily historical windows, using the extracted BOOM source and inherited historical-object exclusion. Existing pools are read only. If BOOM has no later history, optional `USE_ALERCE_FALLBACK=True` checks public ALeRCE photometry for objects without retrieved follow-up; retrieval failures remain separate from physical quality failures. Enable either option with a new RUN_ID. No labels or retraining are needed.

Scores compare uncertainty with later anomalousness, not equivalent classifiers or confirmed labels. AstroMiNN has five broad classes; FLARE bts6 has six. FLARE external context is explicitly missing (NaN), so this is not a reproduction of its full-context paper results.


## 1. Delta setup

Place the notebook in `active-learning/notebooks/presentation/`. Inside a CPU allocation, run `source ~/al_delta_setup.sh` before executing it with `al_python -m nbconvert`. Use the `al-pipeline` kernel with the sibling FLARE clone installed. Start with 4 CPUs and 16 GB RAM; network time depends on the archive. Completed raw batches are reusable after interruption.

All generated products (raw query caches, tables, plots, inference caches, optional fresh pools, and review archives) go to **`/work/nvme/bcrv/dkourtesis/applecider_adaptive`**. Input pools and prediction caches are searched independently on NVMe, HDD and projects, and remain read only. Output storage is never inferred from an input location. Set `AL_COMPARISON_OUTPUT_ROOT` only if explicitly choosing another output directory; there is no automatic fallback to projects or home. A small write probe checks the output directories before querying BOOM.

When executing with nbconvert, also set **`--output-dir`** to the printed report directory, so the executed notebook is written on NVMe. Keep the source notebook in the repository. NVMe is working storage; copy the final review archive back when the run finishes.

The notebook uses a new run ID and never overwrites January's products. Default coverage mode needs no classifier loading or GPU. Compare mode reuses November's saved **sample-level probabilities** when they match the chosen snapshots. If unavailable, enable new inference explicitly; the notebook creates a manifest view referencing those exact existing sample files, without transferring them.

Fresh input and ALeRCE fallback are optional and disabled initially. Fresh input fetches cutouts and can be substantially larger than the default coverage query. Dates, per-day caps, SSO policy, feature code and source manifests are recorded. Capped windows are marked rather than described as complete survey coverage.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gzip, hashlib, inspect, json, os, subprocess, sys, tarfile
import warnings

CPU_THREADS = max(1, int(os.environ.get("SLURM_CPUS_PER_TASK", "4")))
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
try:
    from IPython.display import display
except ImportError:
    display = print

WORKSPACE = Path("/projects/bcrv/dkourtesis/applecider-al-workspace")
AL_REPO = WORKSPACE / "active-learning"
PERSISTENT_ROOT = Path("/projects/bcrv/dkourtesis/applecider_adaptive")
INPUT_MODE = "existing_november"        # alternatively "fresh_boom"
POOL_IDS = ["boom_ztf_2025-11-27_5000", "boom_ztf_2025-11-28_5000",
            "boom_ztf_2025-11-29_5000", "boom_ztf_2025-11_5000"]
RUN_ID = "early_astrominn_later_flare_november_v2"
POOL_ID = "november_earliest_snapshot_view_v2"
EXCLUDE_KNOWN_SSO = True
USE_ALERCE_FALLBACK = False              # new RUN_ID if enabled
DIAGNOSTIC_DAYS = (30.0, 60.0, 100.0)
FRESH_START_UTC = "2025-11-20T00:00:00+00:00"
FRESH_END_UTC = "2025-11-27T00:00:00+00:00"
FRESH_ALERT_CAP_PER_DAY = 2000
RUN_MODE = "coverage"                   # "coverage" first; "compare" second
FOLLOWUP_DAYS = 60.0
ACQUISITION_BUDGET = 25
FLARE_SCHEME = "bts6"
MODEL_NAME = "astrominn_vit135"
MODEL_CLASSES = ("SN I", "SN II", "CV", "AGN", "TDE")
QUERY_BATCH_SIZE = 50
ALERT_QUERY_LIMIT = 20000              # hitting the cap triggers batch splitting
INFERENCE_BATCH_SIZE = 64
DEVICE = "cpu"                        # use "cuda" only inside a GPU allocation
ALLOW_NEW_INFERENCE = False            # cache absence produces a clear stop
MIN_PAIRED_FOR_SUMMARY = 30             # presentation caution, not an eligibility cut

def first_existing(paths, required_child=None):
    for path in paths:
        if (path / required_child).exists() if required_child else path.exists():
            return path
    raise FileNotFoundError("None of these paths exists: " + ", ".join(map(str, paths)))

POOL_ROOTS = [Path("/work/nvme/bcrv/dkourtesis/applecider_adaptive/pools"),
              Path("/work/hdd/bcrv/dkourtesis/applecider_adaptive/pools"),
              PERSISTENT_ROOT / "pools"]
# Outputs are independent of where historical input pools happen to live.
OUTPUT_ROOT = Path(os.environ.get("AL_COMPARISON_OUTPUT_ROOT",
    "/work/nvme/bcrv/dkourtesis/applecider_adaptive")).expanduser()
if not OUTPUT_ROOT.is_absolute():
    raise ValueError("AL_COMPARISON_OUTPUT_ROOT must be an absolute path")
DATA_ROOT = OUTPUT_ROOT  # optional fresh pools are generated here
POOL_DIRS = {}  # populated below; fresh windows are materialized only when explicitly enabled
FLARE_REPO = first_existing([
    WORKSPACE / "flare", WORKSPACE / "applecider-flare",
    Path.home() / "external/applecider-flare",
], "flare/fetch.py")
score_cache_candidates = [root.parent / "comparisons/astrominn_flare_4xpools_v1/astrominn_scores.parquet"
                          for root in POOL_ROOTS]
NOVEMBER_SCORE_CACHE = next((p for p in score_cache_candidates if p.exists()), score_cache_candidates[0])
REGISTRY_PATH = AL_REPO / "configs/delta.toml"
REPORT_DIR = OUTPUT_ROOT / "reports" / RUN_ID
RAW_CACHE_DIR = DATA_ROOT / "comparisons" / RUN_ID / "raw_boom"
import tempfile
for directory in (OUTPUT_ROOT, REPORT_DIR, RAW_CACHE_DIR):
    try:
        directory.mkdir(parents=True, exist_ok=True)
        if not directory.resolve().is_relative_to(OUTPUT_ROOT.resolve()):
            raise ValueError("Output directory points outside OUTPUT_ROOT: " + str(directory))
        with tempfile.TemporaryFile(dir=directory) as probe:
            probe.write(b"0" * 65536)
            probe.flush()
            os.fsync(probe.fileno())
    except OSError as exc:
        raise OSError(exc.errno, "Cannot write notebook outputs here: " + str(directory)
            + ". Check this directory's quota or choose AL_COMPARISON_OUTPUT_ROOT explicitly.",
            str(directory)) from exc

if RUN_MODE not in {"coverage", "compare"}:
    raise ValueError("RUN_MODE must be coverage or compare")
if FOLLOWUP_DAYS <= 0 or ACQUISITION_BUDGET <= 0:
    raise ValueError("Follow-up and acquisition budget must be positive")
if AL_REPO.is_dir():
    sys.path.insert(0, str(AL_REPO / "src"))
sys.path.insert(0, str(FLARE_REPO))
try:
    import flare
    from flare import data as flare_data
    from flare.fetch import to_events, fetch_alerce
    from flare.features import extract_from_array
    from flare.config import resolve_scheme
    from al_pipeline.astronomy.boom.client import BoomClient
    from al_pipeline.core.types import ClassificationPredictions
    from al_pipeline.core.scoring.entropy import EntropyScorer
    from al_pipeline.astronomy.ztf.eligibility import ZTFKnownSolarSystemRule
except ImportError as exc:
    raise ImportError("Use al-pipeline with FLARE installed from the configured clone. "
                      "Missing dependency: " + str(exc)) from exc

if not Path(flare.__file__).resolve().is_relative_to(FLARE_REPO.resolve()):
    raise RuntimeError("Kernel already imported a different FLARE clone; restart it")
plt.rcParams.update({"font.family": "serif", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "savefig.bbox": "tight"})
print("Mode:", RUN_MODE, "| follow-up:", FOLLOWUP_DAYS, "days")
print("Input:", INPUT_MODE, "| SSO exclusion:", EXCLUDE_KNOWN_SSO)
print("FLARE:", FLARE_REPO, "| version:", getattr(flare, "__version__", "unknown"))
print("November prediction cache:", NOVEMBER_SCORE_CACHE, "exists:", NOVEMBER_SCORE_CACHE.exists())
print("Output root:", OUTPUT_ROOT)
print("Raw cache:", RAW_CACHE_DIR)
print("Reports:", REPORT_DIR)

if INPUT_MODE not in {"existing_november", "fresh_boom"}:
    raise ValueError("Unknown INPUT_MODE")


## 2. Freeze the early snapshots; optional fresh historical input

Each source pool retains its original query limits and materialized samples. Duplicate objects across pools are resolved by **minimum packet JD**, then pool ID and candid as deterministic ties. The choice depends only on early snapshots, not later quality or scores. Repeat observations are not independent objects.

For optional fresh input, the configured interval is queried in non-overlapping daily windows with a cap per day. The frozen historical training/validation/test objects are excluded. Known SSO exclusion uses the original packet fields, before inference. Short capped queries remain short samples; this notebook does not pretend a 5,000-alert cap covers a whole month.

This reconstructs availability by measurement JD. It cannot establish the exact original broker ingestion time or recover unavailable archive records. Full-history retrieval is for auditing and later scoring; future detections never enter the early image/metadata input.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)

def git_revision(path):
    result = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                            capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None


if INPUT_MODE == "existing_november":
    for pool_id in POOL_IDS:
        POOL_DIRS[pool_id] = first_existing([root / pool_id for root in POOL_ROOTS], "candidate_pool.csv")
else:
    from al_pipeline.astronomy.data.registry import DatasetRegistry
    from al_pipeline.astronomy.boom.source import BoomZTFSource
    from al_pipeline.core.eligibility.base import EligibilityPolicy
    resolved = DatasetRegistry.from_toml(REGISTRY_PATH).resolve_profile("astrominn_ztf")
    historical_ids = set()
    for manifest_path in (resolved.dataset.train_manifest, resolved.dataset.validation_manifest,
                          resolved.dataset.test_manifest):
        if manifest_path is None or not Path(manifest_path).exists():
            raise FileNotFoundError("Historical exclusion manifest is missing")
        id_column = resolved.dataset.object_id_column
        frame = pd.read_csv(manifest_path, usecols=[id_column], dtype={id_column: str})
        historical_ids.update(frame[id_column].dropna())
    start, stop = pd.Timestamp(FRESH_START_UTC), pd.Timestamp(FRESH_END_UTC)
    if start.tzinfo is None or stop.tzinfo is None or start >= stop:
        raise ValueError("Fresh window requires increasing timezone-aware UTC dates")
    request_config = {"start": FRESH_START_UTC, "end": FRESH_END_UTC,
        "cap_per_day": FRESH_ALERT_CAP_PER_DAY, "max_alert": 10, "excluded_historical_objects": len(historical_ids),
        "historical_manifests": {str(p): sha256(p) for p in
            (resolved.dataset.train_manifest, resolved.dataset.validation_manifest, resolved.dataset.test_manifest)},
        "exclude_sso": EXCLUDE_KNOWN_SSO}
    fresh_config_path = REPORT_DIR / "fresh_request_config.json"
    if fresh_config_path.exists() and json.loads(fresh_config_path.read_text()) != request_config:
        raise RuntimeError("Fresh request changed: use a new RUN_ID")
    write_json(fresh_config_path, request_config)
    fresh_outcomes = []
    cursor = start
    while cursor < stop:
        end = min(cursor + pd.Timedelta(days=1), stop)
        pool_id = RUN_ID + "_" + cursor.strftime("%Y%m%d")
        destination = DATA_ROOT / "pools" / pool_id
        empty_marker = destination / "no_usable_samples.json"
        if empty_marker.exists():
            fresh_outcomes.append(json.loads(empty_marker.read_text()))
            cursor = end
            continue
        if not (destination / "candidate_pool.csv").exists():
            if destination.exists() and any(destination.iterdir()):
                raise RuntimeError("Incomplete fresh pool: inspect it before retrying " + str(destination))
            policy = EligibilityPolicy([ZTFKnownSolarSystemRule(max_distance_arcsec=5.0)]) if EXCLUDE_KNOWN_SSO else None
            try:
                BoomZTFSource(limit=FRESH_ALERT_CAP_PER_DAY, start_jd=cursor.to_julian_date(),
                    end_jd=end.to_julian_date(), excluded_object_ids=historical_ids,
                    max_alert=10, eligibility_policy=policy, save_raw_payloads=False).fetch(
                        pool_id=pool_id, destination=destination)
            except RuntimeError as exc:
                if str(exc) != "BOOM query produced no usable candidate samples.":
                    raise
                destination.mkdir(parents=True, exist_ok=True)
                outcome = {"pool_id": pool_id, "start_utc": cursor.isoformat(), "end_utc": end.isoformat(),
                           "status": "no_usable_candidate_samples"}
                write_json(empty_marker, outcome)
                fresh_outcomes.append(outcome)
                cursor = end
                continue
        POOL_DIRS[pool_id] = destination
        fresh_outcomes.append({"pool_id": pool_id, "start_utc": cursor.isoformat(), "end_utc": end.isoformat(),
                               "status": "materialized"})
        cursor = end
    pd.DataFrame(fresh_outcomes).to_csv(REPORT_DIR / "fresh_window_outcomes.csv", index=False)
    if not POOL_DIRS:
        raise RuntimeError("No fresh window produced usable samples; see fresh_window_outcomes.csv")

source_hashes, indices, inventory = [], [], []
for pool_id, pool_dir in POOL_DIRS.items():
    manifest_path, source_path = pool_dir / "candidate_pool.csv", pool_dir / "source_index.json"
    manifest = pd.read_csv(manifest_path, dtype={"object_id": str, "source_reference": str})
    records = json.loads(source_path.read_text())
    for record in records:
        if not isinstance(record["candid"], (int, str)) or isinstance(record["candid"], bool):
            raise TypeError("Candid must be an exact integer/string")
        record["candid"] = str(record["candid"])
    source = pd.DataFrame(records)
    source["object_id"] = source.object_id.astype(str)
    if set(manifest.pool_id.astype(str)) != {pool_id}:
        raise RuntimeError("Pool ID mismatch")
    merged = source.merge(manifest[["object_id", "alert", "source_reference", "filepath", "survey"]],
        on=["object_id", "alert", "source_reference"], how="left", validate="one_to_one")
    if len(merged) != len(manifest) or merged.filepath.isna().any():
        raise RuntimeError("Source index does not exactly match the sample manifest")
    def sample_path(value):
        path = Path(value)
        if path.is_absolute():
            return str(path)
        candidates = [pool_dir / path, pool_dir / "samples" / path]
        return str(next((p for p in candidates if p.exists()), candidates[0]).resolve())
    merged["filepath"] = merged.filepath.map(sample_path)
    merged["pool_id"] = pool_id
    merged["jd"] = pd.to_numeric(merged.jd, errors="raise")
    if not np.isfinite(merged.jd).all() or not merged.alert.between(0, 10).all():
        raise RuntimeError("Expected finite snapshot times and alert indices <=10")
    indices.append(merged)
    entry = {"pool_id": pool_id, "path": str(pool_dir), "manifest_sha256": sha256(manifest_path),
             "source_index_sha256": sha256(source_path)}
    if (pool_dir / "ingestion_report.json").exists():
        entry["ingestion_report_sha256"] = sha256(pool_dir / "ingestion_report.json")
    source_hashes.append(entry)
    inventory.append({"pool_id": pool_id, "samples": len(merged), "objects": merged.object_id.nunique(),
        "first_snapshot_utc": pd.to_datetime(merged.jd.min(), unit="D", origin="julian", utc=True).isoformat(),
        "last_snapshot_utc": pd.to_datetime(merged.jd.max(), unit="D", origin="julian", utc=True).isoformat()})
all_snapshots = pd.concat(indices, ignore_index=True)
early = all_snapshots.sort_values(["jd", "pool_id", "candid"], kind="stable").drop_duplicates("object_id")
early = early.sort_values("object_id", kind="stable").reset_index(drop=True)
if len(early) < ACQUISITION_BUDGET:
    raise RuntimeError("Budget exceeds the early object universe")
early["late_cutoff_jd"] = early.jd + FOLLOWUP_DAYS
early.to_csv(REPORT_DIR / "early_snapshots_before_eligibility.csv", index=False)
pd.DataFrame(inventory).to_csv(REPORT_DIR / "source_pool_inventory.csv", index=False)
scheme = resolve_scheme(FLARE_SCHEME)
config = {"schema_version": 2, "input_mode": INPUT_MODE, "pool_id": POOL_ID,
    "source_pools": source_hashes, "followup_days": FOLLOWUP_DAYS, "diagnostic_days": list(DIAGNOSTIC_DAYS),
    "budget": ACQUISITION_BUDGET, "snapshot_policy": "earliest_saved_JD_per_object_then_pool_id_candid",
    "exclude_known_sso": EXCLUDE_KNOWN_SSO, "sso_radius_arcsec": 5.0,
    "use_alerce_fallback": USE_ALERCE_FALLBACK, "triggered_query_time_cutoff": None,
    "astro_model": MODEL_NAME, "astro_classes": list(MODEL_CLASSES),
    "flare_scheme": FLARE_SCHEME, "flare_horizon_days": float(scheme.horizon_days),
    "flare_revision": git_revision(FLARE_REPO), "al_revision": git_revision(AL_REPO),
    "flare_context_policy": "all_external_context_missing_NaN",
    "to_events_sha256": hashlib.sha256(inspect.getsource(to_events).encode()).hexdigest(),
    "quality_sha256": hashlib.sha256(inspect.getsource(flare_data.passes_quality).encode()).hexdigest(),
    "features_sha256": sha256(Path(inspect.getfile(extract_from_array))),
    "quality_defaults": {name: getattr(flare_data, name) for name in
        ["MIN_OBS_TOTAL", "MIN_OBS_G", "MIN_OBS_R", "MIN_OBS_I", "MIN_BANDS_OBSERVED"]},
    "color_window_days": float(flare_data.COLOR_WINDOW_DAYS),
    "raw_query_batch_size": QUERY_BATCH_SIZE, "raw_alert_query_limit": ALERT_QUERY_LIMIT}
config_path = REPORT_DIR / "run_config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise RuntimeError("Settings/input hashes changed: choose a new RUN_ID")
write_json(config_path, config)
config_hash = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
print("Saved samples:", len(all_snapshots), "| distinct early objects:", len(early))
display(pd.DataFrame(inventory))


## 3. Full-archive photometry query, with caps and failures audited

Query triggered packets and AUX for **every** early object, with no measurement-time filter in the request. Only the subsequent reconstruction truncates by each object's early and follow-up JD. This permits checking whether later history exists at all, even beyond 60 days.

Full responses are cached by batch with retrieval date and hashes. A capped triggered query splits recursively; a single-object cap stops rather than silently truncating. API failures stop the run and leave completed caches intact. No current photometry is fed into early AstroMiNN inference.

Optional ALeRCE fallback is attempted only when BOOM returned no usable post-snapshot detection. Its rows are cached separately with source and query errors recorded. This can be slow over a large pool. Enabling it changes the experiment configuration and requires a new RUN_ID. Later eligibility is never an acquisition eligibility gate.


In [ ]:
def fetch_alerts_without_truncation(client, object_ids):
    rows = client.find(catalog_name="ZTF_alerts",
        filter_document={"objectId": {"$in": object_ids}},
        limit=ALERT_QUERY_LIMIT,
        projection={"_id": 1, "objectId": 1, "candidate.jd": 1, "candidate.fid": 1,
                    "candidate.magpsf": 1, "candidate.sigmapsf": 1,
                    "candidate.issdiffpos": 1, "candidate.ssnamenr": 1, "candidate.ssdistnr": 1})
    if len(rows) < ALERT_QUERY_LIMIT:
        return rows
    if len(object_ids) == 1:
        raise RuntimeError("Alert cap reached for " + object_ids[0] + "; increase cap with a new RUN_ID")
    midpoint = len(object_ids) // 2
    return (fetch_alerts_without_truncation(client, object_ids[:midpoint])
            + fetch_alerts_without_truncation(client, object_ids[midpoint:]))

aux_by_object, alerts_by_object, raw_receipts = {}, {}, []
client = None
ids = early.object_id.tolist()
for start in range(0, len(ids), QUERY_BATCH_SIZE):
    batch_ids = ids[start:start + QUERY_BATCH_SIZE]
    cache_path = RAW_CACHE_DIR / f"batch_{start:06d}.json.gz"
    if cache_path.exists():
        with gzip.open(cache_path, "rt") as handle:
            payload = json.load(handle)
        if payload["config_hash"] != config_hash or payload["object_ids"] != batch_ids:
            raise RuntimeError("Raw cache belongs to different settings or IDs: " + str(cache_path))
    else:
        if client is None:
            client = BoomClient(query_batch_size=QUERY_BATCH_SIZE)
        aux = client.find_by_ids(catalog_name="ZTF_alerts_aux", ids=batch_ids,
                                 projection={"_id": 1, "prv_candidates": 1})
        alerts = fetch_alerts_without_truncation(client, batch_ids)
        payload = {"config_hash": config_hash, "object_ids": batch_ids,
                   "retrieved_utc": datetime.now(timezone.utc).isoformat(),
                   "query_has_date_cutoff": False, "aux": aux, "alerts": alerts}
        temporary = cache_path.with_name(cache_path.name + ".tmp")
        with gzip.open(temporary, "wt") as handle:
            json.dump(payload, handle, allow_nan=False)
        temporary.replace(cache_path)
    for doc in payload["aux"]:
        oid = str(doc["_id"])
        if oid not in batch_ids or oid in aux_by_object:
            raise RuntimeError("Unexpected or duplicate AUX object " + oid)
        aux_by_object[oid] = doc
    for doc in payload["alerts"]:
        oid = str(doc.get("objectId", ""))
        if oid not in batch_ids:
            raise RuntimeError("Triggered alert returned an unexpected object")
        alerts_by_object.setdefault(oid, []).append(doc)
    raw_receipts.append({"path": str(cache_path), "sha256": sha256(cache_path),
                         "retrieved_utc": payload["retrieved_utc"], "objects_requested": len(batch_ids),
                         "aux_returned": len(payload["aux"]), "alerts_returned": len(payload["alerts"])})
    print(f"Photometry {min(start + QUERY_BATCH_SIZE, len(ids))}/{len(ids)}", flush=True)
write_json(REPORT_DIR / "raw_cache_receipts.json", raw_receipts)
print("Objects with AUX:", len(aux_by_object), "| with triggered alerts:", len(alerts_by_object))


## 4. Early eligibility and follow-up coverage

An exact early packet is required to validate the snapshot and evaluate known-SSO exclusion. Missing early packets are reported as unavailable early evidence. SSO exclusion uses only `ssnamenr` and `ssdistnr` from that early packet; it never uses later classification or light-curve quality.

For the remaining early pool, preserve same-epoch different-band detections, and call FLARE's native event conversion and quality gate. Reconstruct at the early JD, at 30/60/100 days, and at the full available archive. The primary score remains fixed at 60 days. The multi-window table diagnoses availability and should not be used to pick whichever window produces the most attractive score correlation.

Distinguish: no usable post-snapshot photometry retrieved; later photometry beyond the primary window; later quality failure; already eligible; newly eligible. These are archive observations, not claims that an object physically ceased producing alerts. The native 100-day anchor stays at first usable detection, including old detections; age is reported.


In [ ]:
def clean_detections(aux_doc, alert_docs, cutoff_jd):
    candidates = list(aux_doc.get("prv_candidates", []) or [])
    candidates += [doc.get("candidate", {}) or {} for doc in alert_docs]
    detections = {}
    negative_keys = set()
    rejected = 0
    for candidate in candidates:
        try:
            jd, mag, err = [float(candidate[name]) for name in ("jd", "magpsf", "sigmapsf")]
            fid = int(candidate["fid"])
        except (KeyError, TypeError, ValueError, OverflowError):
            rejected += 1
            continue
        if not np.isfinite([jd, mag, err]).all() or err <= 0 or fid not in (1, 2, 3):
            rejected += 1
            continue
        if jd > cutoff_jd:
            continue
        key = (round(jd, 7), fid)
        detections[key] = (jd - 2400000.5, fid, mag, err)
        if str(candidate.get("issdiffpos", "")).lower() in {"f", "false", "0", "-1"}:
            negative_keys.add(key)
        else:
            negative_keys.discard(key)
    return sorted(detections.values()), len(negative_keys), rejected

def event_summary(rows):
    events = to_events(rows, horizon_days=float(scheme.horizon_days))
    if events is None:
        return events, {"n_events": 0, "n_g": 0, "n_r": 0, "n_i": 0, "quality": False}
    if not np.isfinite(events[:, :8]).all():
        raise RuntimeError("Non-finite FLARE event measurements")
    bands = events[:, flare_data.COL["band_id"]].astype(int)
    return events, {"n_events": len(events), "n_g": int((bands == 0).sum()),
                    "n_r": int((bands == 1).sum()), "n_i": int((bands == 2).sum()),
                    "quality": bool(flare_data.passes_quality(events))}


diagnostic_rows, later_events, eligibility_rows, window_rows = [], {}, [], []
fallback_receipts = []
sso_rule = ZTFKnownSolarSystemRule(max_distance_arcsec=5.0)
for object_number, entry in enumerate(early.itertuples(index=False), 1):
    aux = aux_by_object.get(entry.object_id, {})
    alerts = alerts_by_object.get(entry.object_id, [])
    exact = [doc for doc in alerts if str(doc.get("_id")) == entry.candid]
    exact_valid = False
    if len(exact) == 1:
        try:
            exact_valid = abs(float(exact[0]["candidate"]["jd"]) - entry.jd) < 1e-6
        except (KeyError, TypeError, ValueError):
            pass
    early_decision = "accepted"
    if not exact_valid:
        early_decision = "missing_or_mismatched_acquisition_packet"
    elif EXCLUDE_KNOWN_SSO and not sso_rule.evaluate(exact[0]["candidate"]).eligible:
        early_decision = "known_solar_system_object"
    eligibility_rows.append({"object_id": entry.object_id, "early_decision": early_decision})
    full_rows, _, _ = clean_detections(aux, alerts, float("inf"))
    boom_later_n = sum(mjd + 2400000.5 > entry.jd for mjd, *_ in full_rows)
    fallback_status = "not_requested"
    if USE_ALERCE_FALLBACK and early_decision == "accepted" and boom_later_n == 0:
        fallback_path = RAW_CACHE_DIR / (entry.object_id + "_alerce.json")
        if fallback_path.exists():
            fallback = json.loads(fallback_path.read_text())
            if fallback["config_hash"] != config_hash:
                raise RuntimeError("ALeRCE cache configuration mismatch")
        else:
            try:
                rows = fetch_alerce(entry.object_id)
                fallback = {"config_hash": config_hash, "status": "ok", "rows": rows,
                    "retrieved_utc": datetime.now(timezone.utc).isoformat()}
            except Exception as exc:
                fallback = {"config_hash": config_hash, "status": "query_failed", "rows": [],
                    "error_type": type(exc).__name__, "retrieved_utc": datetime.now(timezone.utc).isoformat()}
            write_json(fallback_path, fallback)
        fallback_status = fallback["status"]
        fallback_receipts.append({"path": str(fallback_path), "sha256": sha256(fallback_path),
            "object_id": entry.object_id, "status": fallback_status, "retrieved_utc": fallback["retrieved_utc"]})
        # Add public photometry as auxiliary rows; exact BOOM packets retain duplicate precedence.
        aux = dict(aux)
        aux["prv_candidates"] = list(aux.get("prv_candidates", []) or []) + [
            {"jd": float(mjd) + 2400000.5, "fid": int(fid), "magpsf": mag, "sigmapsf": err}
            for mjd, fid, mag, err in fallback["rows"]]
        full_rows, _, _ = clean_detections(aux, alerts, float("inf"))
    early_rows, _, _ = clean_detections(aux, alerts, float(entry.jd))
    late_rows, negative_n, rejected_n = clean_detections(aux, alerts, float(entry.late_cutoff_jd))
    _, early_summary = event_summary(early_rows)
    events, late_summary = event_summary(late_rows)
    _, full_summary = event_summary(full_rows)
    later_n = sum(mjd + 2400000.5 > entry.jd for mjd, *_ in full_rows)
    within_n = sum(entry.jd < mjd + 2400000.5 <= entry.late_cutoff_jd for mjd, *_ in full_rows)
    if early_decision != "accepted":
        outcome = early_decision
    elif late_summary["quality"]:
        outcome = "already_eligible" if early_summary["quality"] else "became_eligible"
        later_events[entry.object_id] = events
    elif fallback_status == "query_failed" and later_n == 0:
        outcome = "followup_retrieval_failed"
    elif later_n == 0:
        outcome = "no_later_photometry_retrieved"
    elif within_n == 0:
        outcome = "later_photometry_beyond_window"
    else:
        outcome = "fails_later_quality"
    first_jd = full_rows[0][0] + 2400000.5 if full_rows else np.nan
    diagnostic_rows.append({"object_id": entry.object_id,
        "aux_found": entry.object_id in aux_by_object,
        "triggered_alert_count": len(alerts), "exact_acquisition_packet_valid": exact_valid,
        "n_usable_early": len(early_rows), "n_usable_later": len(late_rows),
        "n_usable_full_archive": len(full_rows), "post_snapshot_detections_full_archive": later_n,
        "post_snapshot_detections_within_window": within_n,
        "latest_available_detection_jd": full_rows[-1][0] + 2400000.5 if full_rows else np.nan,
        "first_available_detection_jd": first_jd, "age_days_at_early_snapshot": entry.jd - first_jd,
        "early_within_native_horizon": bool(np.isfinite(first_jd) and entry.jd - first_jd <= scheme.horizon_days),
        "negative_difference_detections_later": negative_n, "invalid_measurement_rows": rejected_n,
        "boom_post_snapshot_detections": boom_later_n, "alerce_status": fallback_status,
        "full_archive_quality": full_summary["quality"], "outcome": outcome,
        **{"early_" + key: value for key, value in early_summary.items()},
        **{"later_" + key: value for key, value in late_summary.items()}})
    if early_decision == "accepted":
        for days in sorted(set(DIAGNOSTIC_DAYS + (FOLLOWUP_DAYS,))):
            rows, _, _ = clean_detections(aux, alerts, float(entry.jd + days))
            _, stats = event_summary(rows)
            window_rows.append({"object_id": entry.object_id, "followup_days": days,
                "post_snapshot_detections": sum(entry.jd < m + 2400000.5 <= entry.jd + days for m, *_ in rows),
                "quality": stats["quality"], "became_eligible": stats["quality"] and not early_summary["quality"]})
    if object_number % 500 == 0:
        print(f"Reconstructed {object_number}/{len(early)} objects", flush=True)
write_json(REPORT_DIR / "alerce_cache_receipts.json", fallback_receipts)
early_eligibility = pd.DataFrame(eligibility_rows)
early_eligibility.to_csv(REPORT_DIR / "early_eligibility.csv", index=False)
all_coverage = early.merge(pd.DataFrame(diagnostic_rows), on="object_id", validate="one_to_one")
all_coverage.to_csv(REPORT_DIR / "all_snapshot_coverage.csv", index=False)
early = early.loc[early.object_id.isin(early_eligibility.loc[early_eligibility.early_decision.eq("accepted"), "object_id"])].copy()
early.to_csv(REPORT_DIR / "early_snapshots.csv", index=False)
coverage = all_coverage.loc[all_coverage.object_id.isin(early.object_id)].copy()
coverage.to_csv(REPORT_DIR / "photometry_coverage.csv", index=False)
counts = coverage.outcome.value_counts().rename_axis("outcome").reset_index(name="objects")
counts["fraction_of_early_pool"] = counts.objects / len(early) if len(early) else 0.
counts.to_csv(REPORT_DIR / "coverage_summary.csv", index=False)
windows = pd.DataFrame(window_rows, columns=["object_id", "followup_days", "post_snapshot_detections", "quality", "became_eligible"])
windows.to_csv(REPORT_DIR / "followup_window_records.csv", index=False)
window_summary = windows.groupby("followup_days").agg(objects=("object_id", "size"),
    objects_with_followup=("post_snapshot_detections", lambda x: int((x > 0).sum())),
    flare_eligible=("quality", "sum"), newly_eligible=("became_eligible", "sum")).reset_index()
window_summary.to_csv(REPORT_DIR / "followup_window_summary.csv", index=False)
print("Early eligibility decisions:")
display(early_eligibility.early_decision.value_counts())
print("Accepted early objects:", len(early), "| raw saved samples:", len(all_snapshots))
display(counts)
display(window_summary)
paired_available = len(later_events)
if paired_available < MIN_PAIRED_FOR_SUMMARY:
    warnings.warn(f"Only {paired_available} primary-window pairs; descriptive pilot only.")
fig, ax = plt.subplots(figsize=(7.6, 4.0))
coverage_labels = {"became_eligible": "Eligible after follow-up", "already_eligible": "Already eligible early",
    "fails_later_quality": "Later quality fails", "followup_retrieval_failed": "Follow-up query failed",
    "no_later_photometry_retrieved": "No later photometry retrieved",
    "later_photometry_beyond_window": "Later photometry beyond window"}
if len(counts):
    ax.barh(counts.outcome.map(coverage_labels), counts.objects, color="#315377")
    ax.invert_yaxis()
    for i, n in enumerate(counts.objects):
        ax.text(n + max(1, len(early)*.006), i, str(n), va="center")
    ax.set_xlim(0, max(counts.objects) * 1.15)
ax.set_xlabel("Objects passing early eligibility")
ax.set_title(f"Follow-up coverage after {FOLLOWUP_DAYS:g} days; N={len(early):,}")
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(REPORT_DIR / f"followup_coverage.{extension}", dpi=220)
plt.show()
if RUN_MODE == "coverage":
    print("COVERAGE COMPLETE. Examine no-followup and multi-window counts before compare mode.")


## 5. Early entropy acquisition on the accepted early universe

Reuse the earlier November **sample-level** probability cache only when it matches each chosen `(pool_id, object_id, alert, candid)` snapshot. The old audit established its output semantics; provenance here records the cache hash and explicitly identifies inherited model provenance. New probabilities are validated, and `EntropyScorer` handles logits/probabilities once.

If a matching cache is unavailable, `ALLOW_NEW_INFERENCE=True` invokes the extracted frozen AstroMiNN adapter over a manifest view with one chosen original file per object. No sample files are copied. Missing snapshots are not quietly replaced by an object's latest alert. Top-k is ranked over the entire accepted early universe, independently of later eligibility, with deterministic object-ID ties.


In [ ]:
astro, inference_provenance = None, None
if RUN_MODE == "compare":
    if len(early) < ACQUISITION_BUDGET:
        raise RuntimeError("Too few accepted early objects for the configured budget")
    local_cache = REPORT_DIR / "early_inference.npz"
    cached_rows = None
    if INPUT_MODE == "existing_november" and NOVEMBER_SCORE_CACHE.exists() and not local_cache.exists():
        cached = pd.read_parquet(NOVEMBER_SCORE_CACHE)
        expected_cols = ["astro_p_" + label for label in MODEL_CLASSES]
        if not set(["pool_id", "object_id", "alert", "candid"] + expected_cols).issubset(cached.columns):
            warnings.warn("November cache schema lacks required provenance/probability columns")
        else:
            cached["object_id"] = cached.object_id.astype(str)
            # Parquet candid must preserve integer precision; unsafe floats are rejected.
            if pd.api.types.is_float_dtype(cached.candid):
                raise RuntimeError("Legacy candid column is floating point; recompute rather than risk a rounded join")
            cached["candid"] = cached.candid.astype(str)
            cached_rows = early[["pool_id", "object_id", "alert", "candid"]].merge(cached,
                on=["pool_id", "object_id", "alert", "candid"], how="left", validate="one_to_one")
            if cached_rows[expected_cols].isna().any().any():
                warnings.warn("November cache does not cover every chosen snapshot")
                cached_rows = None
    if cached_rows is not None:
        object_ids = cached_rows.object_id.to_numpy()
        alerts = cached_rows.alert.to_numpy(dtype=np.int64)
        predictions = ClassificationPredictions(values=cached_rows[expected_cols].to_numpy(float), output_type="probabilities")
        inference_provenance = {"path": str(NOVEMBER_SCORE_CACHE), "sha256": sha256(NOVEMBER_SCORE_CACHE),
            "metadata_status": "inherited_November_sample_probability_cache", "output_type": "probabilities",
            "caveat": "Original model provenance inherited from the audited notebook-02 cache"}
    else:
        if not local_cache.exists():
            if not ALLOW_NEW_INFERENCE:
                raise FileNotFoundError("Matching November probabilities unavailable. Set NOVEMBER_SCORE_CACHE correctly "
                    "or enable ALLOW_NEW_INFERENCE explicitly; missing snapshots are not substituted")
            from al_pipeline.astronomy.astrominn.inference.run import run_inference
            view_dir = REPORT_DIR / "prepared_snapshot_pool"
            (view_dir / "samples").mkdir(parents=True, exist_ok=True)
            view = early[["survey", "object_id", "alert", "filepath", "source_reference"]].copy()
            view.insert(0, "pool_id", POOL_ID)
            missing = [path for path in view.filepath if not Path(path).is_file()]
            if missing:
                raise FileNotFoundError("Original sample file missing: " + missing[0])
            view.to_csv(view_dir / "candidate_pool.csv", index=False)
            write_json(view_dir / "ingestion_report.json", {"source": "reference_view_no_new_ingestion",
                       "samples": len(view), "source_pools": source_hashes})
            result = run_inference(model=MODEL_NAME, pool_path=view_dir, device=DEVICE,
                                   batch_size=INFERENCE_BATCH_SIZE, num_workers=0)
            np.savez_compressed(local_cache, pool_id=np.asarray(POOL_ID),
                object_ids=np.asarray(result.object_ids, dtype=str), alerts=np.asarray(result.alerts, dtype=np.int64),
                values=result.predictions.values, output_type=np.asarray(result.predictions.output_type))
        with np.load(local_cache, allow_pickle=False) as payload:
            if str(payload["pool_id"].item()) != POOL_ID:
                raise RuntimeError("Wrong local inference cache pool")
            object_ids, alerts = payload["object_ids"].astype(str), payload["alerts"].astype(np.int64)
            predictions = ClassificationPredictions(values=np.asarray(payload["values"]),
                output_type=str(payload["output_type"].item()))
        inference_provenance = {"path": str(local_cache), "sha256": sha256(local_cache),
            "metadata_status": "generated_by_this_notebook", "output_type": predictions.output_type}
    if len(object_ids) != len(set(object_ids)) or set(object_ids) != set(early.object_id):
        raise RuntimeError("Inference must cover the accepted early pool exactly once")
    if predictions.values.shape != (len(object_ids), len(MODEL_CLASSES)):
        raise RuntimeError("Prediction class dimension mismatch")
    if not np.array_equal(alerts, early.set_index("object_id").loc[object_ids, "alert"].to_numpy()):
        raise RuntimeError("Inference used different snapshots")
    p = predictions.as_probabilities()
    astro = pd.DataFrame({"object_id": object_ids, "alert": alerts,
        "astro_entropy": EntropyScorer().score(predictions),
        "astro_predicted_class": np.asarray(MODEL_CLASSES)[p.argmax(axis=1)], "astro_max_probability": p.max(axis=1)})
    for j, label in enumerate(MODEL_CLASSES):
        astro["astro_p_" + label] = p[:, j]
    astro = astro.sort_values(["astro_entropy", "object_id"], ascending=[False, True], kind="stable")
    astro["early_entropy_rank"] = np.arange(1, len(astro) + 1)
    astro["selected_early"] = astro.early_entropy_rank <= ACQUISITION_BUDGET
    astro.to_csv(REPORT_DIR / "early_entropy_acquisition.csv", index=False)
    write_json(REPORT_DIR / "inference_provenance.json", inference_provenance)
    display(astro.head(ACQUISITION_BUDGET)[["object_id", "alert", "astro_entropy", "astro_predicted_class"]])
else:
    print("Coverage mode: inference and acquisition skipped.")


## 6. Later FLARE features and classifier-native energy

All eligible objects are scored, including early nonselections. External host, redshift, Gaia/WISE and positional context columns are explicitly left as NaN. FLARE documents this missing-context path; the report lists exactly which inputs are missing. No current catalog context is introduced into a historical replay.

We call `anomaly_energy` directly. If the clone contains FLARE's dedicated `ad_space` booster, that is the energy model; otherwise FLARE falls back to top-classifier logits and the report names that fallback. Higher energy is ranked first. This is neither FLARE entropy nor an anomaly-label decision.


In [ ]:
flare_scores = None
flare_provenance = None
if RUN_MODE == "compare" and paired_available > 0:
    clf = flare.load_classifier(FLARE_SCHEME)
    if not hasattr(clf, "anomaly_energy"):
        raise RuntimeError("Configured FLARE classifier has no anomaly_energy API")
    energy_booster = getattr(clf, "ad_space", None)
    energy_source = "dedicated_ad_space" if energy_booster is not None else "top_classifier_fallback"
    if energy_booster is None:
        energy_booster = clf.top
        warnings.warn("This clone uses top-classifier logits for energy; dedicated ad_space was not loaded")
    required = list(dict.fromkeys(list(clf.feature_names) + list(energy_booster.feature_name())))
    external_cols = set(scheme.external_cols)
    feature_rows = []
    for oid, events in later_events.items():
        features = extract_from_array(events)
        feature_rows.append({"object_id": oid, **features})
    if feature_rows:
        features = pd.DataFrame(feature_rows).set_index("object_id")
        missing_columns = [name for name in required if name not in features]
        unexpected_missing = set(missing_columns) - external_cols
        if unexpected_missing:
            raise RuntimeError("Missing light-curve feature columns: " + str(sorted(unexpected_missing)))
        X = features.reindex(columns=required).replace([np.inf, -np.inf], np.nan)
        energy = np.asarray(clf.anomaly_energy(X), dtype=float)
        labels = np.asarray(clf.predict(X))
        if energy.shape != (len(X),) or not np.isfinite(energy).all():
            raise RuntimeError("FLARE returned invalid energy scores")
        flare_scores = pd.DataFrame({"object_id": X.index,
            "flare_energy": energy, "flare_predicted_class": labels,
            "n_missing_flare_features": X.isna().sum(axis=1).to_numpy()})
        X.to_csv(REPORT_DIR / "later_flare_features.csv", index_label="object_id")
    else:
        missing_columns = required
        flare_scores = pd.DataFrame(columns=["object_id", "flare_energy", "flare_predicted_class",
                                               "n_missing_flare_features"])
    flare_scores.to_csv(REPORT_DIR / "later_flare_scores.csv", index=False)
    flare_provenance = {"scheme": FLARE_SCHEME, "energy_source": energy_source,
        "energy_features": list(energy_booster.feature_name()),
        "classification_features": list(clf.feature_names),
        "external_context_policy": "missing_NaN", "missing_input_columns": missing_columns,
        "model_files": {key: {"path": str(path), "sha256": sha256(path)}
                        for key, path in scheme.model_paths.items() if Path(path).is_file()}}
    write_json(REPORT_DIR / "flare_provenance.json", flare_provenance)
    display(flare_scores.head())
else:
    if RUN_MODE == "compare":
        flare_scores = pd.DataFrame(columns=["object_id", "flare_energy", "flare_predicted_class", "n_missing_flare_features"])
        flare_provenance = {"energy_source": "not_loaded_no_pairable_objects"}
        flare_scores.to_csv(REPORT_DIR / "later_flare_scores.csv", index=False)
        write_json(REPORT_DIR / "flare_provenance.json", flare_provenance)
    print("FLARE classifier scoring skipped (coverage mode or no pairable objects).")


## 7. Paired results, coverage and two distinct overlap questions

**Original acquisition:** of the full-pool top-25 objects, how many can FLARE score later? We keep the original selection and its denominator. Within eligible objects, compare the surviving selection to the same number of highest-energy objects. This is a conditional comparison, not a full-pool FLARE ranking.

**Matched rankings:** on a shared eligible cohort, rerank early entropy and compare its top-k to FLARE's top-k. This is a retrospective ranking diagnostic and must not be presented as the original early acquisition. Report both all later-eligible objects and the subset that became eligible after acquisition.

Spearman correlation uses the entire eligible cohort, including nonselected objects; small or constant-score cohorts do not support a meaningful coefficient. Class predictions are diagnostic annotations, not ground truth, and the two model taxonomies differ.


In [ ]:
def correlation_record(frame):
    if len(frame) < 3 or frame.astro_entropy.nunique() < 2 or frame.flare_energy.nunique() < 2:
        return {"n": len(frame), "spearman_rho": None, "spearman_p_value": None}
    result = spearmanr(frame.astro_entropy, frame.flare_energy)
    return {"n": len(frame), "spearman_rho": float(result.statistic),
            "spearman_p_value": float(result.pvalue)}

def overlap_records(frame, cohort_name):
    rows = []
    entropy_order = frame.sort_values(["astro_entropy", "object_id"], ascending=[False, True])
    energy_order = frame.sort_values(["flare_energy", "object_id"], ascending=[False, True])
    # Do not report k=N, where overlap is guaranteed by construction.
    for k in sorted(set([5, 10, ACQUISITION_BUDGET])):
        if k >= len(frame):
            continue
        a, b = set(entropy_order.head(k).object_id), set(energy_order.head(k).object_id)
        common = len(a & b)
        rows.append({"cohort": cohort_name, "cohort_n": len(frame), "k": k,
            "intersection": common, "overlap_fraction": common / k,
            "jaccard": common / len(a | b), "random_expected_overlap_fraction": k / len(frame)})
    return rows

summary = {"mode": RUN_MODE, "pool_id": POOL_ID, "early_pool_objects": len(early),
    "followup_days": FOLLOWUP_DAYS, "later_pairable_objects": paired_available,
    "saved_sample_rows": len(all_snapshots), "distinct_objects_before_eligibility": len(all_coverage),
    "early_eligibility_outcomes": {str(k): int(v) for k, v in early_eligibility.early_decision.value_counts().items()},
    "objects_with_any_retrieved_followup": int((coverage.post_snapshot_detections_full_archive > 0).sum()),
    "full_archive_quality_objects": int(coverage.full_archive_quality.sum()),
    "coverage_outcomes": {str(k): int(v) for k, v in coverage.outcome.value_counts().items()},
    "context_policy": "missing external context", "warnings": []}
if paired_available < MIN_PAIRED_FOR_SUMMARY:
    summary["warnings"].append("Small primary-window paired cohort; descriptive pilot only")
if not len(coverage) or not (coverage.post_snapshot_detections_full_archive > 0).any():
    summary["warnings"].append("No post-snapshot photometry retrieved; inspect archive coverage before comparing")
paired = None
if RUN_MODE == "compare":
    all_objects = coverage.merge(astro.drop(columns="alert"), on="object_id", validate="one_to_one")
    all_objects = all_objects.merge(flare_scores, on="object_id", how="left", validate="one_to_one")
    all_objects.to_csv(REPORT_DIR / "all_object_results.csv", index=False)
    paired = all_objects.loc[all_objects.flare_energy.notna()].copy()
    paired.to_csv(REPORT_DIR / "paired_objects.csv", index=False)
    cohorts = {"all_later_eligible": paired,
               "became_eligible_later": paired.loc[paired.outcome.eq("became_eligible")]}
    correlations = {name: correlation_record(frame) for name, frame in cohorts.items()}
    overlap = pd.DataFrame([row for name, frame in cohorts.items() for row in overlap_records(frame, name)],
        columns=["cohort", "cohort_n", "k", "intersection", "overlap_fraction", "jaccard",
                 "random_expected_overlap_fraction"])
    overlap.to_csv(REPORT_DIR / "matched_ranking_overlap.csv", index=False)
    selected = paired.loc[paired.selected_early]
    surviving_n = len(selected)
    energy_selected = set(paired.sort_values(["flare_energy", "object_id"], ascending=[False, True])
                                  .head(surviving_n).object_id)
    surviving_ids = set(selected.object_id)
    common = len(surviving_ids & energy_selected)
    original_overlap = {"original_budget": ACQUISITION_BUDGET,
        "selected_scoreable_later": surviving_n, "eligible_comparison_universe": len(paired),
        "selected_followup_coverage": surviving_n / ACQUISITION_BUDGET,
        "intersection_with_energy_top_surviving_n": common,
        "conditional_overlap_fraction": common / surviving_n if surviving_n else None,
        "full_budget_intersection_fraction": common / ACQUISITION_BUDGET,
        "ranking_trivial": bool(surviving_n and surviving_n == len(paired))}
    selected_coverage = all_objects.loc[all_objects.selected_early, "outcome"].value_counts()
    summary.update({"correlations": correlations, "original_acquisition": original_overlap,
        "selected_coverage_outcomes": {str(k): int(v) for k, v in selected_coverage.items()},
        "energy_source": flare_provenance["energy_source"],
        "inference_metadata_status": inference_provenance["metadata_status"]})
    if len(paired) < MIN_PAIRED_FOR_SUMMARY:
        summary["warnings"].append("Small paired cohort; descriptive pilot only")
    if surviving_n == 0:
        summary["warnings"].append("No original entropy-selected objects could be scored by FLARE later")
    if original_overlap["ranking_trivial"]:
        summary["warnings"].append("Surviving selection spans the eligible universe; original-selection overlap is trivial")
    if inference_provenance["metadata_status"] == "legacy_model_metadata_unavailable":
        summary["warnings"].append("Legacy model metadata unavailable; provenance requires verification")
    display(pd.DataFrame(correlations).T)
    display(pd.DataFrame([original_overlap]))
    display(overlap)
write_json(REPORT_DIR / "comparison_summary.json", summary)


## 8. Presentation figures

The main figure puts early entropy on the horizontal axis and later energy on the vertical axis. Gray points are early nonselections; blue circles are early selections that became eligible later; blue squares are early selections FLARE could already score. A hollow black circle identifies newly eligible nonselections. Every plotted point is one object.

Coverage stays visible beside the comparison. A second panel separates already-eligible from newly eligible objects; a third shows matched-cohort ranking overlap when enough objects are available. All figures are exported as PNG and PDF. The notebook still completes and reports coverage if zero objects are pairable.


In [ ]:
def plot_pairs(ax, frame, title):
    for selected_flag, outcome, marker, face, edge, label in [
        (False, "already_eligible", "s", "#bbbbbb", "#bbbbbb", "Nonselected; already eligible"),
        (False, "became_eligible", "o", "none", "#777777", "Nonselected; eligible later"),
        (True, "already_eligible", "s", "#315377", "#315377", "Selected; already eligible"),
        (True, "became_eligible", "o", "#315377", "#315377", "Selected; eligible later"),
    ]:
        subset = frame.loc[frame.selected_early.eq(selected_flag) & frame.outcome.eq(outcome)]
        if len(subset):
            ax.scatter(subset.astro_entropy, subset.flare_energy, s=35 if not selected_flag else 70,
                marker=marker, facecolors=face, edgecolors=edge, linewidths=.8,
                alpha=.75 if not selected_flag else 1., label=label, zorder=3 if selected_flag else 2)
    ax.set_xlabel("Early AstroMiNN entropy [nats]")
    ax.set_ylabel("Later FLARE anomaly energy")
    ax.set_title(title)
    record = correlation_record(frame)
    rho_text = "undefined" if record["spearman_rho"] is None else f'{record["spearman_rho"]:.2f}'
    ax.text(.03, .97, f'N = {len(frame)}; Spearman rho = {rho_text}',
            transform=ax.transAxes, va="top", fontsize=10,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": .95, "pad": 3})
    if len(frame):
        ax.legend(loc="best", fontsize=8)

if RUN_MODE == "compare":
    if len(paired):
        fig, ax = plt.subplots(figsize=(8.0, 4.7))
        plot_pairs(ax, paired, f"Early uncertainty → energy after {FOLLOWUP_DAYS:g} days")
        fig.text(.5, -.025,
            f'Original top-{ACQUISITION_BUDGET}: {len(selected)}/{ACQUISITION_BUDGET} scoreable later '
            f'| FLARE bts6; external context missing', ha="center", fontsize=9)
        fig.tight_layout()
        for extension in ("png", "pdf"):
            fig.savefig(REPORT_DIR / f"early_entropy_later_energy.{extension}", dpi=240)
        plt.show()
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
        plot_pairs(axes[0], paired.loc[paired.outcome.eq("already_eligible")], "Already eligible at the early snapshot")
        plot_pairs(axes[1], paired.loc[paired.outcome.eq("became_eligible")], "Became eligible during follow-up")
        fig.tight_layout()
        for extension in ("png", "pdf"):
            fig.savefig(REPORT_DIR / f"eligibility_cohort_comparison.{extension}", dpi=220)
        plt.show()
    else:
        print("No pairable objects: coverage plot is the appropriate result.")
    if len(overlap):
        fig, ax = plt.subplots(figsize=(7.2, 3.8))
        for name, group in overlap.groupby("cohort", sort=False):
            ax.plot(group.k, group.overlap_fraction, marker="o", label=name.replace("_", " "))
            ax.plot(group.k, group.random_expected_overlap_fraction, linestyle=":", color="#999999",
                    label="Random expectation (" + name.replace("_", " ") + ")")
        ax.set(xlabel="Top-k within the same eligible cohort", ylabel="Intersection / k", ylim=(0, 1.05))
        ax.legend(fontsize=8)
        fig.tight_layout()
        for extension in ("png", "pdf"):
            fig.savefig(REPORT_DIR / f"matched_ranking_overlap.{extension}", dpi=220)
        plt.show()


## 9. Concrete examples and review bundle

The table below supplies objects for an optional slide example. It uses within-cohort ranks to find agreement and disagreement; it does not assign confirmed astrophysical labels. Each row retains the early candid/JD, alert index, band counts, follow-up cutoff, predictions and both scores. The complete CSV contains all paired objects.

The review archive contains only the modest report products—not large raw photometry, images, model weights or credentials. Raw compressed query caches remain on Delta, with their locations and hashes in `raw_cache_receipts.json`. Rerunning compare mode uses the same cached photometry. Save the executed notebook separately, or copy it into `REPORT_DIR` before rerunning this cell if you want it included in the archive.


In [ ]:
if RUN_MODE == "compare" and len(paired):
    example_table = paired.copy()
    n = len(example_table)
    example_table["entropy_rank_fraction"] = example_table.astro_entropy.rank(ascending=False, method="average") / n
    example_table["energy_rank_fraction"] = example_table.flare_energy.rank(ascending=False, method="average") / n
    example_table["rank_gap"] = example_table.energy_rank_fraction - example_table.entropy_rank_fraction
    examples = []
    seen = set()
    for description, frame in [
        ("Original entropy selection, strongest later energy", example_table.loc[example_table.selected_early]
            .sort_values("flare_energy", ascending=False)),
        ("High early uncertainty relative to later anomaly rank", example_table.sort_values("rank_gap", ascending=False)),
        ("High later anomaly rank relative to early uncertainty", example_table.sort_values("rank_gap")),
    ]:
        for _, row in frame.iterrows():
            if row.object_id not in seen:
                examples.append({"example": description, **row.to_dict()})
                seen.add(row.object_id)
                break
    examples = pd.DataFrame(examples)
    examples.to_csv(REPORT_DIR / "example_objects.csv", index=False)
    display(examples[["example", "object_id", "outcome", "selected_early", "astro_entropy", "flare_energy",
                      "early_n_events", "later_n_events", "later_n_g", "later_n_r"]])

readme = f'''# {RUN_ID}
Mode: {RUN_MODE}. Original early pool: {len(early)} objects.
Input mode: {INPUT_MODE}. Early known-SSO exclusion: {EXCLUDE_KNOWN_SSO}.
Follow-up cutoff: each saved early snapshot JD + {FOLLOWUP_DAYS:g} days.
FLARE: {FLARE_SCHEME}; native first-{scheme.horizon_days:g}-day window from first available detection.
External FLARE context is missing (NaN); this is a photometry-only/missing-context comparison.
Source pools have query caps; actual returned snapshot dates are recorded in source_pool_inventory.csv.
Inspect comparison_summary.json, coverage_summary.csv and photometry_coverage.csv first.
Full raw photometry stays on Delta; see raw_cache_receipts.json for locations/hashes.
Confirmed labels were not queried. Neither prediction nor energy establishes an astrophysical class.
Matched ranking overlap reranks within eligible cohorts; it is distinct from original AL selection.
'''
(REPORT_DIR / "README.txt").write_text(readme)
archive_path = REPORT_DIR.parent / (RUN_ID + "_" + RUN_MODE + "_review.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    coverage_products = {"early_snapshots.csv", "run_config.json", "raw_cache_receipts.json",
        "photometry_coverage.csv", "coverage_summary.csv", "comparison_summary.json", "README.txt",
        "followup_coverage.png", "followup_coverage.pdf", "source_pool_inventory.csv", "early_eligibility.csv",
        "early_snapshots_before_eligibility.csv", "all_snapshot_coverage.csv", "alerce_cache_receipts.json",
        "followup_window_records.csv", "followup_window_summary.csv", "fresh_request_config.json", "fresh_window_outcomes.csv"}
    for path in sorted(REPORT_DIR.iterdir()):
        if path.is_file() and path.suffix in {".csv", ".json", ".txt", ".png", ".pdf", ".ipynb"}:
            if RUN_MODE == "coverage" and path.name not in coverage_products and path.suffix != ".ipynb":
                continue
            archive.add(path, arcname=RUN_ID + "/" + path.name)
print("Review archive:", archive_path)
print(json.dumps(summary, indent=2))


## Suggested slide after we inspect the real output

**Title:** Early uncertainty and later anomaly ranking

- Main figure: `early_entropy_later_energy.pdf` (or the newly-eligible panel if that cohort is sufficiently populated).
- Small statistics: paired object count; original top-25 follow-up coverage; conditional overlap or Spearman rho when meaningful.
- Footnote: fixed 60-day follow-up; FLARE native quality gate; external context missing; one point per object.
- Interpretation should follow the results: agreement, complementary rankings, or insufficient follow-up coverage. Do not promise a correlation or usable anomaly-label yield before the run.

**Implementation sources:** new `al_pipeline.core.scoring.entropy`, `al_pipeline.astronomy.boom.client`, and `al_pipeline.astronomy.astrominn.inference.run`; November notebooks 02/03 and the extracted BOOM source; FLARE `fetch.to_events`, `data.passes_quality`, `features.extract_from_array`, `hierarchical.HierarchicalFlare.anomaly_energy`. FLARE API inspection for this notebook used repository commit `e95a86360f25f860ea275d33546e34dd40ca5d6a`; runtime provenance records your actual clone and model hashes.
